In [ ]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt


from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


def virtual_exp(r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12):

    complexity = sum(1 for x in [r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12] if x != 0)
    cost = r1+r2+r3+r4+r5+r6+r7+r8+r9+r10+r11+r12
    performance = 0.3*r1*(1+r2) - 0.5*r3*r4 + r5**2 + 0.8*r9 - r10*r11 + 0.2*r12
    return {'complexity': complexity, 'cost': cost, 'performance': performance}


# generation strategy
gs = GenerationStrategy(
    steps=[
        GenerationStep(
            model=Models.SOBOL,
            num_trials=8,  # how many sobol trials to perform (rule of thumb: 2 * number of params)
            model_kwargs={"seed": 0},
        ),
        GenerationStep(
            model=Models.SAASBO,
            num_trials=-1,
            model_kwargs={},
        ),
    ]
)

# initialize the AxClient
ax_client = AxClient(generation_strategy=gs)

# create the design space and objective space
ax_client.create_experiment(
    parameters=[

        {"name": "r1", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r2", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r3", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r4", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r5", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r6", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r7", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r8", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r9", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r10", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r11", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "r12", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"}],


    objectives={
        'complexity': ObjectiveProperties(minimize=True, threshold=5),
        'cost': ObjectiveProperties(minimize=True, threshold=0.5),
        'performance': ObjectiveProperties(minimize=False),
    },


    parameter_constraints=[
#        "c1 + c2 + c3 + c4 + c5 + c6 + c7 + c8 + c9 + c10 + c11 + c12 <= 8.0",  # example of a sum constraint, which may be redundant/unintended if composition_constraint is also selected
#        "r1 + r2 + r3 + r4 + r5 + r6 + r7 + r8 + r9 + r10 + r11 + r12 <= 1.0",  # example of a sum constraint, which may be redundant/unintended if composition_constraint is also selected
    ],
)


batch_size = 1

for i in range(30):

    parameterizations, optimization_complete = ax_client.get_next_trials(batch_size)
    for trial_index, parameterization in list(parameterizations.items()):
        # extract parameters

        r1 = parameterization["r1"]
        r2 = parameterization["r2"]
        r3 = parameterization["r3"]
        r4 = parameterization["r4"]
        r5 = parameterization["r5"]
        r6 = parameterization["r6"]
        r7 = parameterization["r7"]
        r8 = parameterization["r8"]
        r9 = parameterization["r9"]
        r10 = parameterization["r10"]
        r11 = parameterization["r11"]
        r12 = parameterization["r12"]

        results = virtual_exp(r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12)
        ax_client.complete_trial(trial_index=trial_index, raw_data=results)
        


                

[INFO 04-28 12:18:05] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 04-28 12:18:05] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='r1', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r2', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r3', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r4', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r5', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r6', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r7', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r8', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r9', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='r10', parameter_type=FLOAT, range=[0.0, 1.0]), RangeP

In [ ]:
objectives = ax_client.objective_names
df = ax_client.get_trials_data_frame()
pareto_results = ax_client.get_pareto_optimal_parameters()

In [ ]:
df

In [ ]:
import seaborn as sns

for col in objectives:
    fig = plt.figure()
    sns.scatterplot(data=df, x='trial_index', y=col, hue='generation_node', palette='deep')

In [ ]:
Models.